# Set your priorities

> Add a criterion, choose weights and see whether they change the plan

::: {.callout-note appearance="simple" icon=false}
**How-to guide: get one job done.** It assumes that you know the basics. New to trufl? Take the [tutorial](../tutorials/01_getting_started.ipynb) first. [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

**You need:** `areas`, a `state` with one row per area, and a numeric value per area for anything you want to add. Here that is `population`, indexed by `loc_id`.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

population = pd.Series(np.random.default_rng(7).integers(50, 5000, len(areas)), index=areas.index, name='Population')

## 1. Add your criterion

In [ ]:
criteria = state.astype(float).join(population)
assert criteria['Population'].notna().all(), 'some areas have no population'
criteria.head(3).round(2)

,Max,Min,Standard Deviation,Count,Prior,Population
loc_id,,,,,,
0,0.11,0.11,0.0,1.0,0.11,4727
1,0.09,0.09,0.0,1.0,0.13,3144
2,0.17,0.17,0.0,1.0,0.16,3436


If your criterion is a class, such as land use, map each class to a number first.

## 2. Give every column a direction and a weight

Write them by column name, so that they cannot slip out of order. Keep one line for each column that you have. Before a first round that is `Prior`, and any column that you added.

In [ ]:
import pandas as pd
from trufl.optimizer import Optimizer

priorities = pd.DataFrame({
    'Max':                (True,  0.20),
    'Min':                (True,  0.10),
    'Standard Deviation': (True,  0.10),
    'Count':              (False, 0.20),
    'Prior':              (True,  0.20),
    'Population':         (True,  0.20),
}, index=['high is urgent', 'weight']).T

assert list(priorities.index) == list(criteria.columns), 'the columns differ from the priorities'
priorities

,high is urgent,weight
Max,True,0.2
Min,True,0.1
Standard Deviation,True,0.1
Count,False,0.2
Prior,True,0.2
Population,True,0.2


## 3. Rank

In [ ]:
def rank(weights):
    weights = weights.astype(float)
    return Optimizer(criteria).get_rank(is_benefit_x=priorities['high is urgent'].astype(bool).tolist(),
                                        w_vector=(weights / weights.sum()).tolist(),
                                        n_method='LINEAR1', s_method='CP')

ranks = rank(priorities['weight'])

# for the guides that follow
state = criteria
directions = priorities['high is urgent'].astype(bool).tolist()
weights = (priorities['weight'].astype(float) / priorities['weight'].sum()).tolist()

ranks.head()

,rank
loc_id,
70,1
26,2
64,3
54,4
99,5


If you get an error about negative values or zero, use `n_method='LINEAR2'`.

## 4. Test how much each weight matters

Double one weight at a time, and count how many areas leave the top quarter:

In [ ]:
top = len(criteria) // 4
changed = {}
for name in criteria.columns:
    doubled = priorities['weight'].copy()
    doubled[name] *= 2
    changed[name] = top - len(set(ranks.index[:top]) & set(rank(doubled).index[:top]))

pd.Series(changed, name=f'areas leaving the top {top}').sort_values(ascending=False)

Count                 24
Max                    7
Prior                  7
Standard Deviation     6
Population             5
Min                    3
Name: areas leaving the top 25, dtype: int64

Discuss the weights at the top of this list with the people who will answer for the plan. The others change little.

## 5. Record the decision

Save `priorities` with the plan, with the date and the names of those who agreed on it.

Next: [fit the plan to your budget](04_fit_budget.ipynb).